# Gemma 3 4B — 1,100-example accuracy experiment

Select a **GPU runtime**. Upload the accompanying `experiment_1100_bundle.zip` when requested. This runs accuracy-only LoRA with 4-bit loading, using the accepted training labels and 100 human validation labels. A six-example smoke run must finish before the full run. Baseline and trained metrics use the same validation set. Full answers are retained; GPU memory requirements depend on sequence lengths. No results exist until the cells run.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
archive = "experiment_1100_bundle.zip"
if archive not in uploaded:
    raise ValueError("Upload experiment_1100_bundle.zip")
ROOT = Path("/content/experiment_1100")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for item in bundle.infolist():
        if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe archive path")
    bundle.extractall(ROOT)
print(ROOT)


In [ ]:
%pip install -q -r /content/experiment_1100/requirements-gpu.txt


In [ ]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import subprocess, sys
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
SMOKE = ROOT / ("smoke_" + RUN_ID)
FULL = ROOT / ("full_" + RUN_ID)
script = ROOT / "input/scripts/run_experiment.py"
subprocess.run([sys.executable, "-u", str(script), "--output-dir", str(SMOKE), "--smoke", "--four-bit"], check=True)
assert (SMOKE / "COMPLETED").exists()


In [ ]:
subprocess.run([sys.executable, "-u", str(script), "--output-dir", str(FULL), "--four-bit"], check=True)
assert (FULL / "COMPLETED").exists()


In [ ]:
import json
print(json.dumps(json.loads((FULL / "metrics/baseline_vs_trained.json").read_text()), indent=2))


In [ ]:
import shutil
results_zip = shutil.make_archive(str(ROOT / ("results_" + RUN_ID)), "zip", FULL)
files.download(results_zip)
